# Notebook 02 — Self-Attention

**Goal:** Build scaled dot-product self-attention from scratch and understand what the attention matrix means.

We'll focus on the essential pipeline:

```
input embeddings X
   │
   ▼  linear projections
Q, K, V
   │
   ▼  QKᵀ / √d_k
attention scores
   │
   ▼  softmax
attention weights
   │
   ▼  weights @ V
contextualised output
```

---

### Contents
1. [Scaled dot-product attention](#1)
2. [A worked example](#2)
3. [Why scaling matters](#3)
4. [Causal masking](#4)
5. [Key takeaways](#5)


In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('=' * 60)
print('  Notebook 02 — Self-Attention')
print('=' * 60)
print(f'  PyTorch : {torch.__version__}')
print('=' * 60)


<a id='1'></a>
## 1 — Scaled dot-product attention

For self-attention, each token produces three vectors:

- **Q (query)**: what this token is looking for
- **K (key)**: what this token offers
- **V (value)**: the information this token carries

The core equation is:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(rac{QK^T}{\sqrt{d_k}}ight)V
$$

where:
- $QK^T$ gives a similarity score between every query and every key
- dividing by $\sqrt{d_k}$ keeps the softmax well behaved
- the softmax turns each row into a probability distribution
- multiplying by $V$ gives the weighted combination of value vectors


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q:    (..., seq_len, d_k)
    K:    (..., seq_len, d_k)
    V:    (..., seq_len, d_v)
    mask: (..., seq_len, seq_len), True = block position
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))

    weights = F.softmax(scores, dim=-1)
    weights = torch.nan_to_num(weights, nan=0.0)
    output = weights @ V
    return output, weights


<a id='2'></a>
## 2 — A worked example

We'll use a tiny sequence of four tokens and set **`Q = K = V = X`** so we can focus purely on the attention computation.


In [ ]:
tokens = ['The', 'cat', 'sat', 'down']

X = torch.tensor([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])

Q = X.clone()
K = X.clone()
V = X.clone()

output, weights = scaled_dot_product_attention(Q, K, V)

print('Input X:')
print(X)
print()
print('Attention weights:')
print(weights)
print()
print('Output:')
print(output)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    weights.numpy(),
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=tokens,
    yticklabels=tokens,
    linewidths=0.4,
    linecolor='white',
    square=True,
    ax=ax,
)
ax.set_title('Self-Attention Weights', fontweight='bold')
ax.set_xlabel('Key token')
ax.set_ylabel('Query token')
plt.tight_layout()
plt.show()

print('Each row shows where one token is looking.')
print('Rows sum to 1.0 because softmax turns them into probabilities.')


In [ ]:
print('=' * 60)
print('Worked example: output for token "cat"')
print('=' * 60)

row = 1
for tok, w in zip(tokens, weights[row]):
    print(f'  attends to {tok:>4s}: {w.item():.3f}')

manual = sum(weights[row, j] * V[j] for j in range(len(tokens)))
print('
Weighted sum of value vectors:')
print(manual)
print('
Matches output row:', torch.allclose(manual, output[row]))


<a id='3'></a>
## 3 — Why scaling matters

Without scaling, the dot products grow with dimension. That makes the softmax too peaky, which hurts training because gradients become very small.


In [ ]:
seq_len = 6
d_k = 128
Q_big = torch.randn(seq_len, d_k)
K_big = torch.randn(seq_len, d_k)

raw_scores = Q_big @ K_big.T
scaled_scores = raw_scores / math.sqrt(d_k)

w_raw = F.softmax(raw_scores, dim=-1)
w_scaled = F.softmax(scaled_scores, dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, w, title in [
    (axes[0], w_raw, 'Without scaling'),
    (axes[1], w_scaled, 'With scaling'),
]:
    sns.heatmap(w.numpy(), cmap='Blues', vmin=0, vmax=1, cbar=False, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')

plt.tight_layout()
plt.show()

print(f'Max attention weight without scaling: {w_raw.max().item():.4f}')
print(f'Max attention weight with scaling:    {w_scaled.max().item():.4f}')


<a id='4'></a>
## 4 — Causal masking

For a GPT-style decoder, token *i* must not look at future tokens *j > i*.

So we apply an **upper-triangular mask** before the softmax.


In [ ]:
seq_len = len(tokens)
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)

masked_output, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print('Causal mask (1 = blocked):')
print(causal_mask.int())
print()
print('Masked attention weights:')
print(masked_weights)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.heatmap(weights.numpy(), annot=True, fmt='.2f', cmap='Blues',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.4, linecolor='white', square=True, ax=axes[0])
axes[0].set_title('Unmasked', fontweight='bold')
axes[0].set_xlabel('Key token')
axes[0].set_ylabel('Query token')

sns.heatmap(masked_weights.numpy(), annot=True, fmt='.2f', cmap='Blues',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.4, linecolor='white', square=True, ax=axes[1])
axes[1].set_title('Causal mask applied', fontweight='bold')
axes[1].set_xlabel('Key token')
axes[1].set_ylabel('Query token')

plt.tight_layout()
plt.show()


<a id='5'></a>
## 5 — Key takeaways

- Self-attention compares every query with every key
- The softmax turns each row into a probability distribution
- The output is a weighted sum of value vectors
- Scaling by $\sqrt{d_k}$ is important for stable training
- Causal masking prevents the model from looking into the future

In the next notebook, we'll run **multiple attention heads in parallel**.
